In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q02/annotated/checkpoints/post_cell_4.pickle

trying: ['load_part']
me:  1
trying: ['load_supplier']
me:  5
trying: ['pd']
me:  0
trying: ['partsupp']
me:  3
trying: ['load_partsupp']
me:  3
trying: ['orig_output']
me:  2
trying: ['load_nation']
me:  7
trying: ['region']
me:  9
trying: ['nation']
me:  7
trying: ['DATA_ROOT']
me:  0
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['supplier']
me:  5
trying: ['load_region']
me:  9
trying: ['part']


me:  1
Declaring variable pd
Declaring variable DATA_ROOT
Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable load_part
Declaring variable part
Declaring variable orig_output
Declaring variable partsupp
Declaring variable load_partsupp
Declaring variable load_supplier
Declaring variable supplier
Declaring variable load_nation
Declaring variable nation
Declaring variable region
Declaring variable load_region


In [4]:
%%RecordEvent
%%time
### cell 5 ###

# Pre-filter auxiliary tables
region_europe = region[region['R_NAME'] == 'EUROPE'][['R_REGIONKEY']]
supplier_sel = supplier[[
    'S_SUPPKEY','S_NAME','S_ADDRESS','S_NATIONKEY',
    'S_PHONE','S_ACCTBAL','S_COMMENT'
]]
partsupp_sel = partsupp[['PS_PARTKEY','PS_SUPPKEY','PS_SUPPLYCOST']]
part_sel = part[
    (part['P_SIZE'] == 15) & part['P_TYPE'].str.endswith('BRASS')
][['P_PARTKEY','P_MFGR']]

# Chain merges in one GPU-resident DataFrame
df = (
    nation[['N_NATIONKEY','N_NAME','N_REGIONKEY']]
    .merge(region_europe, left_on='N_REGIONKEY', right_on='R_REGIONKEY', how='inner')
    .merge(supplier_sel,  left_on='N_NATIONKEY', right_on='S_NATIONKEY', how='inner')
    .merge(partsupp_sel,  left_on='S_SUPPKEY',   right_on='PS_SUPPKEY',  how='inner')
    .merge(part_sel,      left_on='PS_PARTKEY',  right_on='P_PARTKEY',   how='inner')
)

# Filter to rows with the minimum supply cost per part and reset index
df = (
    df[df['PS_SUPPLYCOST'] == df.groupby('P_PARTKEY')['PS_SUPPLYCOST'].transform('min')]
      .reset_index(drop=True)
)

# Final projection and ordering
total = df[
    ['S_ACCTBAL','S_NAME','N_NAME','P_PARTKEY','P_MFGR',
     'S_ADDRESS','S_PHONE','S_COMMENT']
].sort_values(
    by=['S_ACCTBAL','N_NAME','S_NAME','P_PARTKEY'],
    ascending=[False, True, True, True]
)


CPU times: user 84.2 ms, sys: 35.3 ms, total: 120 ms
Wall time: 127 ms


In [5]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q02/rewritten/o4_mini_high/checkpoints/post_cell_5_try_3.pickle

migration speed (bps): 807949986.1725783
---------------------------
variables to migrate:
load_region 144
region 48
region_europe 48
total 48
load_partsupp 144
supplier 48
partsupp 48
partsupp_sel 48
load_supplier 144
load_part 144
part 48
df 48
supplier_sel 48
part_sel 48
STORAGE_OPTS 64
tpch_parent 54
DATA_ROOT 80
orig_output 16
nation 48
load_nation 144
pd 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/tpch/notebooks/q02/rewritten/o4_mini_high/checkpoints/post_cell_5_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables []
Intermediate variables ['part']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables ['partsupp']
Future variables ['part']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables ['supplier']
Future variables ['partsupp', 'part']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables ['nation']
Future variables ['partsupp', 'supplier', 'part']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables ['region']
Future variables ['nation', 'partsupp', 'supplier', 'part']
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['region_europe', 'partsupp_sel', 'part_sel', 'supplier_sel', 'total', 'df']
Future variables []
Modified dataframes
Saved cell execution i

In [7]:

with open("/home/dias-benchmarks/tpch/notebooks/q02/opt_cell_exec_info_5_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[5], f)


In [8]:
opt_output = Out.get(4)

In [9]:
total_opt = total
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q02/annotated/checkpoints/post_cell_5.pickle
assert compare_df(total_opt, total)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['supplier_filtered']
me:  11
trying: ['load_region']
me:  9
trying: ['region']
me:  9
trying: ['part_filtered']
me:  11
trying: ['s_r_n_merged']
me:  11
trying: ['load_partsupp']
me:  3
trying: ['supplier']
me:  5
trying: ['partsupp']
me:  3
trying: ['nation_filtered']
me:  11
trying: ['load_supplier']
me:  5
trying: ['r_n_merged']
me:  11
trying: ['merged_df']
me:  11
trying: ['load_part']
me:  1
trying: ['part']
me:  1
trying: ['total']
me:  11
trying: ['region_filtered']
me:  11
trying: ['STORAGE_OPTS']
me:  0
trying: ['tpch_parent']
me:  0
trying: ['min_values']
me:  11
trying: ['DATA_ROOT']
me:  0
trying: ['ps_s_r_n_merged']


me:  11
trying: ['partsupp_filtered']
me:  11
trying: ['orig_output']
me:  2
trying: ['nation']
me:  7
trying: ['load_nation']
me:  7
trying: ['pd']
me:  0


Declaring variable STORAGE_OPTS
Declaring variable tpch_parent
Declaring variable DATA_ROOT
Declaring variable pd
Declaring variable load_part
Declaring variable part
Declaring variable orig_output
Declaring variable load_partsupp
Declaring variable partsupp
Declaring variable supplier
Declaring variable load_supplier
Declaring variable nation
Declaring variable load_nation
Declaring variable load_region
Declaring variable region
Declaring variable supplier_filtered
Declaring variable part_filtered
Declaring variable s_r_n_merged
Declaring variable nation_filtered
Declaring variable r_n_merged
Declaring variable merged_df
Declaring variable total
Declaring variable region_filtered
Declaring variable min_values
Declaring variable ps_s_r_n_merged
Declaring variable partsupp_filtered


ValueError: Index of df1 and df2 do not match